# 03 - Machine Learning Fundamentals with Scikit-learn

This notebook introduces a simple classification workflow for business data.

Goal: predict whether a financial observation is stressed (`StressFlag = 1`) or normal (`StressFlag = 0`).

Beginner concepts covered:

- Feature columns: input values used by the model.
- Target column: the value the model tries to predict.
- Train/test split: train on one part of the data and test on unseen rows.
- Accuracy, precision, recall, and confusion matrix.
- Model reliability and limitations.

In [ ]:
# Install required libraries for a fresh notebook environment such as Google Colab.
import sys
import subprocess

for package in ["pandas", "numpy", "matplotlib", "seaborn", "scikit-learn", "sqlalchemy", "pyodbc", "python-dotenv"]:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    except Exception as exc:
        print(f"Package install skipped or failed for {package}: {exc}")

In [ ]:
import os
from pathlib import Path
from urllib.parse import quote_plus

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    import pyodbc
except Exception as exc:
    pyodbc = None
    print("pyodbc unavailable; SQL Server extraction will be skipped:", exc)

try:
    from dotenv import load_dotenv
    from sqlalchemy import create_engine
except Exception as exc:
    load_dotenv = None
    create_engine = None
    print("SQLAlchemy/dotenv unavailable; SQL Server extraction will be skipped:", exc)

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
ENV_FILE = ".env"
sns.set_theme(style="whitegrid")

In [ ]:
# Shared data loader. It tries SQL Server first, then uses generated fallback data.
def build_sample_indicator_data():
    institutions = [
        ("CBL", "Central Bank Liquidity Desk", "Maseru", "Central Bank", 920_000_000, 310_000_000, 0.345, 0.218, 0.020, 1),
        ("MCB", "Maseru Commercial Bank", "Maseru", "Commercial Bank", 610_000_000, 520_000_000, 0.278, 0.161, 0.048, 2),
        ("LMB", "Leribe Microfinance Bank", "Leribe", "Microfinance", 185_000_000, 171_000_000, 0.235, 0.139, 0.073, 3),
        ("QFB", "Quthing Finance Bank", "Quthing", "Commercial Bank", 275_000_000, 251_000_000, 0.242, 0.148, 0.067, 4),
        ("BDB", "Butha-Buthe Development Bank", "Butha-Buthe", "Development Bank", 330_000_000, 298_000_000, 0.255, 0.154, 0.061, 5),
        ("MFI", "Mafeteng Inclusion Finance", "Mafeteng", "Microfinance", 145_000_000, 139_000_000, 0.218, 0.128, 0.086, 6),
    ]
    rows = []
    observation_id = 1
    for day_number, observation_date in enumerate(pd.date_range("2026-04-01", periods=60, freq="D")):
        for code, name, region, inst_type, deposits, loans, liquidity, capital, npl, weight in institutions:
            liquidity_ratio = round(liquidity + ((day_number % 10) * 0.0021) - (0.014 if weight in (3, 6) else 0) - (0.018 if 33 <= day_number <= 42 else 0), 4)
            capital_ratio = round(capital + ((day_number % 8) * 0.0014) - (0.011 if weight == 6 else 0), 4)
            npl_ratio = round(npl + ((day_number % 9) * 0.0018) + (0.010 if 33 <= day_number <= 42 else 0), 4)
            credit_growth = round(0.0310 + ((day_number % 12) * 0.0016) - (npl * 0.0800), 4)
            stress_flag = int(liquidity_ratio < 0.2300 or capital_ratio < 0.1300 or npl_ratio >= 0.0850 or credit_growth < 0.0280)
            rows.append({
                "ObservationID": observation_id,
                "ObservationDate": observation_date,
                "InstitutionCode": code,
                "InstitutionName": name,
                "Region": region,
                "InstitutionType": inst_type,
                "LiquidityRatio": liquidity_ratio,
                "CapitalAdequacyRatio": capital_ratio,
                "NplRatio": npl_ratio,
                "TransactionVolume": 850 + (weight * 95) + (day_number * 8) + ((day_number % 6) * 35),
                "TransactionValueLSL": (deposits * 0.028) + (day_number * 185_000) + (weight * 75_000) + ((day_number % 7) * 430_000),
                "CreditGrowthRate": credit_growth,
                "InflationRate": round(0.0470 + ((day_number % 11) * 0.0007), 4),
                "InterbankRate": round(0.0820 + ((day_number % 7) * 0.0009), 4),
                "StressFlag": stress_flag,
            })
            observation_id += 1
    return pd.DataFrame(rows)

def load_indicator_data():
    query = """
    SELECT ObservationID, ObservationDate, InstitutionCode, InstitutionName, Region, InstitutionType,
           LiquidityRatio, CapitalAdequacyRatio, NplRatio, TransactionVolume, TransactionValueLSL,
           CreditGrowthRate, InflationRate, InterbankRate, CAST(StressFlag AS INT) AS StressFlag
    FROM m5.DailyFinancialIndicators
    ORDER BY ObservationDate, InstitutionCode;
    """
    try:
        if pyodbc is None or load_dotenv is None or create_engine is None:
            raise RuntimeError("SQL Server packages are unavailable")
        load_dotenv(BASE_DIR / ENV_FILE)
        driver = os.getenv("DB_DRIVER", "ODBC Driver 18 for SQL Server")
        server = os.getenv("DB_SERVER", "localhost,1433")
        database = os.getenv("DB_NAME", "TrainingDB")
        user = os.getenv("DB_USER", "sa")
        password = os.getenv("DB_PASSWORD", "StrongPassw0rd!2026")
        trusted = os.getenv("DB_TRUSTED", "no").lower() in ("yes", "true", "1")
        parts = [f"DRIVER={{{driver}}}", f"SERVER={server}", f"DATABASE={database}", "Encrypt=yes", "TrustServerCertificate=yes", "Connection Timeout=5"]
        parts.append("Trusted_Connection=yes" if trusted else f"UID={user};PWD={password}")
        engine = create_engine(f"mssql+pyodbc:///?odbc_connect={quote_plus(';'.join(parts) + ';')}")
        data = pd.read_sql(query, engine, parse_dates=["ObservationDate"])
        source = "SQL Server"
    except Exception as exc:
        print("Using generated sample data. SQL Server unavailable:", exc)
        data = build_sample_indicator_data()
        source = "generated fallback data"
    print("Data source:", source)
    return data

df = load_indicator_data()
display(df.head())
display(df["StressFlag"].value_counts(normalize=True).rename("share"))

## 1. Prepare Features and Target

`X` contains input features. `y` contains the target we want to predict.

We use `stratify=y` in the train/test split so the stress/normal class balance is similar in both sets.

In [ ]:
features = [
    "LiquidityRatio", "CapitalAdequacyRatio", "NplRatio", "TransactionVolume",
    "TransactionValueLSL", "CreditGrowthRate", "InflationRate", "InterbankRate",
]

X = df[features]
y = df["StressFlag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training class balance:")
print(y_train.value_counts(normalize=True))

## 2. Train a Simple Logistic Regression Model

Logistic regression is a beginner-friendly classification model. Here it predicts a probability of stress.

`StandardScaler` standardizes numeric features so large-value columns do not dominate smaller-ratio columns.

In [ ]:
model = Pipeline([
    ("scale", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, zero_division=0)
recall = recall_score(y_test, predictions, zero_division=0)

print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print("\nClassification report:")
print(classification_report(y_test, predictions, target_names=["Normal", "Stress"], zero_division=0))

print("Interpretation:")
print("Accuracy is the overall correct classification rate. Recall for Stress is important when missing stressed cases is costly.")

## 3. Confusion Matrix

A confusion matrix shows correct and incorrect classifications.

- True normal: actual normal predicted normal.
- False stress: actual normal predicted stress.
- False normal: actual stress predicted normal.
- True stress: actual stress predicted stress.

In [ ]:
matrix = confusion_matrix(y_test, predictions)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title("Stress Classifier Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_xticklabels(["Normal", "Stress"])
ax.set_yticklabels(["Normal", "Stress"], rotation=0)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "stress_classifier_confusion_matrix.png", dpi=160)
plt.show()

## 4. Feature Coefficients and Reliability Notes

Coefficients show how each standardized feature contributes to the model's prediction.

For this training model, reliability should be judged by:

- Test-set accuracy, precision, and recall.
- Whether the confusion matrix shows unacceptable missed stress cases.
- Whether the model was tested on data it did not train on.
- Whether the dataset is large and realistic enough for production use.

In [ ]:
coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": model.named_steps["classifier"].coef_[0],
}).sort_values("coefficient", ascending=False)

metrics = pd.DataFrame([{
    "model": "LogisticRegression",
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "train_rows": len(X_train),
    "test_rows": len(X_test),
}])

coefficients.to_csv(OUTPUT_DIR / "stress_model_coefficients.csv", index=False)
metrics.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)

display(coefficients)
display(metrics)

if recall < 0.75:
    print("Reliability warning: stress recall is below 75%; review features, thresholds, or training data.")
else:
    print("Reliability note: stress recall is acceptable for this guided practice dataset, but production validation would need more data.")